In [2]:
import os
import torch
import open_clip
import pandas as pd
import torch.nn as nn
from tqdm.auto import tqdm
from PIL import Image
from accelerate import Accelerator
import torchvision.transforms as T
from torch.optim import AdamW
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from typing import Dict, Any, List, Optional
from torch.utils.data.distributed import DistributedSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import pytrec_tgt as pe
from collections.abc import Mapping, Sequence

import warnings
warnings.filterwarnings("ignore")

In [3]:
import sys
sys.path.append('/home/jovyan/clip/clip_tt')

from model.clip_tt_attention_fusion import CLIPTwoTower as attention_fusion_model
from model.clip_tt_mlp_fusion import CLIPTwoTower as mlp_fusion_model
from model.clip_tt_moe_bilinear_fusion import CLIPTwoTower as moe_bilinear_fusion_model
from model.clip_tt_moe_fusion import CLIPTwoTower as moe_fusion_model
from model.clip_tt_moe_fusion_query_projection import CLIPTwoTower as moe_fusion_query_projection_model

In [4]:
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "query": row["query"],
            "title": row["title"],
            "image": row["local_path"],
        }


class Collator:
    def __init__(self, image_size: int = 224, clip_model_name: str = "ViT-B-32"):
        self.tokenizer = open_clip.get_tokenizer(clip_model_name)
        self.tf = T.Compose([
            T.Resize(image_size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),  # CLIP defaults
                        std=(0.26862954, 0.26130258, 0.27577711)),
        ])

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        queries = [b["query"] for b in batch]
        titles   = [b["title"] for b in batch]
        paths   = [b["image"] for b in batch]

        imgs = []
        for p in paths:
            with Image.open(p) as im:
                imgs.append(self.tf(im.convert("RGB")))
        images = torch.stack(imgs, dim=0)

        query_ids = self.tokenizer(queries)
        title_ids = self.tokenizer(titles)


        return {
            "query_ids": query_ids,
            "title_ids": title_ids,
            "images": images,
        }


def build_eval_loader(
    df: pd.DataFrame,
    batch_size: int = 128,
    num_workers: int = 8,
    image_size: int = 224,
    ddp_world_size: int = 1,
    ddp_rank: int = 0,
    clip_model_name: str = "ViT-B-32",
    train: bool = True,
):
    dataset = ImageDataset(df)
    collate = Collator(image_size=image_size, clip_model_name=clip_model_name)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collate,
        shuffle=False,
    )
    return loader

In [5]:
def move_to_device(x, device):
    if torch.is_tensor(x):
        return x.to(device, non_blocking=True)
    if isinstance(x, Mapping):
        return {k: move_to_device(v, device) for k, v in x.items()}
    if isinstance(x, Sequence) and not isinstance(x, (str, bytes)):
        return type(x)(move_to_device(v, device) for v in x)
    return x

@torch.no_grad()
def evaluate_with_dataloader(df, model, dataloader, device, new_col="cosine"):
    all_cosines = []
    df_deep = df.copy()
    pbar = tqdm(total=len(dataloader))
    for batch in dataloader:
        batch = move_to_device(batch, device)
        cosines = model.evaluate(batch)     # (B, d)

        all_cosines.extend(cosines.cpu().tolist())
        pbar.update(1)

    df_deep[new_col] = all_cosines
    return df_deep


In [6]:
df = pd.read_parquet("/home/jovyan/image_tt/data/relevance_eval.parquet")
dataloader = build_eval_loader(df=df, batch_size=512, ddp_world_size=1, ddp_rank=0)

device = "cuda" if torch.cuda.is_available() else "cpu"
attention_fusion_clip = attention_fusion_model(unlock_layers=0).to(device)
ckpt = torch.load("/home/jovyan/clip/clip_tt/checkpoints/attention_fusion_freeze_all/attention_fusion_freeze_all_6.pt", map_location=device)
attention_fusion_clip.load_state_dict(ckpt)
attention_fusion_clip.eval()
attention_fusion_relevance_df = evaluate_with_dataloader(df, attention_fusion_clip, dataloader, device)

100%|██████████| 775/775 [08:38<00:00,  1.50it/s]


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
mlp_fusion_clip = mlp_fusion_model(unlock_layers=0).to(device)
ckpt = torch.load("/home/jovyan/clip/clip_tt/checkpoints/mlp_fusion_freeze_all/mlp_fusion_freeze_all_6.pt", map_location=device)
mlp_fusion_clip.load_state_dict(ckpt)
mlp_fusion_clip.eval()
mlp_fusion_relevance_df = evaluate_with_dataloader(df, mlp_fusion_clip, dataloader, device)

100%|██████████| 775/775 [08:33<00:00,  1.51it/s]


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
moe_bilinear_fusion_clip = moe_bilinear_fusion_model(unlock_layers=0).to(device)
ckpt = torch.load("/home/jovyan/clip/clip_tt/checkpoints/moe_bilinear_fusion_freeze_all/moe_bilinear_fusion_freeze_all_6.pt", map_location=device)
moe_bilinear_fusion_clip.load_state_dict(ckpt)
moe_bilinear_fusion_clip.eval()
moe_bilinear_fusion_relevance_df = evaluate_with_dataloader(df, moe_bilinear_fusion_clip, dataloader, device)

100%|██████████| 775/775 [08:34<00:00,  1.51it/s]


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
moe_fusion_clip = moe_fusion_model(unlock_layers=0).to(device)
ckpt = torch.load("/home/jovyan/clip/clip_tt/checkpoints/moe_fusion_freeze_all/moe_fusion_freeze_all_6.pt", map_location=device)
moe_fusion_clip.load_state_dict(ckpt)
moe_fusion_clip.eval()
moe_fusion_relevance_df = evaluate_with_dataloader(df, moe_fusion_clip, dataloader, device)

100%|██████████| 775/775 [08:34<00:00,  1.51it/s]


In [10]:
MEASURES = [pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(df, score_col="cosine", label_col="relevancy_score", query_col="query"):
    rating_dict = df.groupby(query_col).apply(lambda x: dict(zip(x['tcin'], x[label_col]))).to_dict()
    result_dict = df.groupby(query_col).apply(lambda x: list(zip(x['tcin'], x[score_col], x['tcin']))).to_dict()

    result = pe.ModelResult(result_dict)
    rating = pe.QueryRating(rating_dict)

    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(result, query_ratings=rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    
    return evaluate_results

attention_fusion_ndcg = evaluate_ndcg(attention_fusion_relevance_df)
print("Attention Fusion NDCG:", attention_fusion_ndcg)
mlp_fusion_ndcg = evaluate_ndcg(mlp_fusion_relevance_df)
print("MLP Fusion NDCG:", mlp_fusion_ndcg)
moe_bilinear_fusion_ndcg = evaluate_ndcg(moe_bilinear_fusion_relevance_df)
print("MoE Bilinear Fusion NDCG:", moe_bilinear_fusion_ndcg)
moe_fusion_ndcg = evaluate_ndcg(moe_fusion_relevance_df)
print("MoE Fusion NDCG:", moe_fusion_ndcg)

Attention Fusion NDCG: {'avg_ndcg': {'avg_ndcg': {1: 0.9119932998324959, 3: 0.9040767859373031, 9: 0.9026542447660069, 24: 0.908843224972921, 200: 0.948206552765889}}}
MLP Fusion NDCG: {'avg_ndcg': {'avg_ndcg': {1: 0.9103517587939699, 3: 0.9050202264787192, 9: 0.9030590751467036, 24: 0.9093856802563077, 200: 0.9486186654302791}}}
MoE Bilinear Fusion NDCG: {'avg_ndcg': {'avg_ndcg': {1: 0.9083752093802345, 3: 0.9040061996254322, 9: 0.9026907428621495, 24: 0.9093108215966701, 200: 0.9484048579760492}}}
MoE Fusion NDCG: {'avg_ndcg': {'avg_ndcg': {1: 0.9074371859296483, 3: 0.9033304932638405, 9: 0.9017722962815967, 24: 0.9081249014584812, 200: 0.9478100323372621}}}


In [6]:
df = pd.read_parquet("/home/jovyan/image_tt/data/relevance_eval.parquet")
dataloader = build_eval_loader(df=df, batch_size=512, ddp_world_size=1, ddp_rank=0)

device = "cuda" if torch.cuda.is_available() else "cpu"
moe_fusion_projection_clip = moe_fusion_query_projection_model(unlock_layers=0).to(device)
ckpt = torch.load("/home/jovyan/clip/clip_tt/checkpoints/moe_fusion_with_projection_freeze_all/moe_fusion_with_projection_freeze_all_6.pt", map_location=device)
moe_fusion_projection_clip.load_state_dict(ckpt)
moe_fusion_projection_clip.eval()
moe_fusion_projection_relevance_df = evaluate_with_dataloader(df, moe_fusion_projection_clip, dataloader, device)

100%|██████████| 775/775 [08:37<00:00,  1.50it/s]


In [7]:
MEASURES = [pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(df, score_col="cosine", label_col="relevancy_score", query_col="query"):
    rating_dict = df.groupby(query_col).apply(lambda x: dict(zip(x['tcin'], x[label_col]))).to_dict()
    result_dict = df.groupby(query_col).apply(lambda x: list(zip(x['tcin'], x[score_col], x['tcin']))).to_dict()

    result = pe.ModelResult(result_dict)
    rating = pe.QueryRating(rating_dict)

    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(result, query_ratings=rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    
    return evaluate_results

moe_fusion_projection_ndcg = evaluate_ndcg(moe_fusion_projection_relevance_df)
print("MoE Fusion Projection NDCG:", moe_fusion_projection_ndcg)

MoE Fusion Projection NDCG: {'avg_ndcg': {'avg_ndcg': {1: 0.910787269681742, 3: 0.9063902580071164, 9: 0.9037576836512179, 24: 0.9097460011960075, 200: 0.9486933846873198}}}
